# Analysis I - Observed workloads
**Evidence class: observational.** This notebook measures properties of a published sensor log. It does not measure internal server packets, watts, or causal energy savings.

## Questions and methods
1. **How variable are logged arrivals?** Count records by calendar day and source hour; examine per-sensor interarrival gaps.
2. **How complete is the source?** Audit timestamps, roster IDs, missing channels and each sensor's representation before selecting replay intervals.
3. **Can these data establish communication or energy savings?** No: the fields contain readings and log times, not service edges, bytes or power.

**Original source:** [Intel Lab Data](https://db.lcs.mit.edu/labdata/labdata.html), Intel Berkeley Research Lab, 28 February-5 April 2004. The published log has about 2.3 million records from 54 sensors. The [dataset register](sources.md) records the other sources planned for service graph and power context. Download the original `data.txt.gz` to `../data/raw/intel/` relative to this notebook.


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

SOURCE = 'https://db.lcs.mit.edu/labdata/labdata.html'
PATH = Path('../data/raw/intel/data.txt.gz')
assert PATH.exists(), f'Download the original archive from {SOURCE} to {PATH}'
sha256 = hashlib.sha256(PATH.read_bytes()).hexdigest()
print('Input SHA-256:', sha256)


Input SHA-256: d99288c8f406ca6604d359ceaa0d8adfffa79e7095061a1e27dc4399f48c7225


## Parse and audit
The original file uses a single space delimiter. Empty channel values must remain empty; splitting on arbitrary whitespace can shift columns without an error. Keep the raw record count and exclusions visible. Source times have no asserted timezone.


In [2]:
columns = ['date','time','epoch','moteid','temperature','humidity','light','voltage']
raw = pd.read_csv(PATH, sep=' ', names=columns, header=None, compression='gzip', low_memory=False)
raw['timestamp'] = pd.to_datetime(raw['date'].astype(str) + ' ' + raw['time'].astype(str), errors='coerce')
for col in columns[2:]: raw[col] = pd.to_numeric(raw[col], errors='coerce')
audit = pd.Series({
    'raw records':len(raw), 'invalid timestamps':int(raw.timestamp.isna().sum()),
    'missing sensor ID':int(raw.moteid.isna().sum()),
    'out-of-roster ID':int((raw.moteid.notna() & ~raw.moteid.between(1,54)).sum()),
    'at least one missing channel':int(raw[['temperature','humidity','light','voltage']].isna().any(axis=1).sum())})
display(audit.to_frame('records'))
events = raw.loc[raw.timestamp.notna() & raw.moteid.between(1,54), ['timestamp','moteid']].sort_values('timestamp',kind='stable')
display(Markdown(f'**Technical:** {len(events):,} records have valid timestamps and roster IDs. Missing values or long gaps cannot be attributed to network loss without packet logs.  \n**Plain language:** We can count what was recorded; we cannot tell why something is absent.'))


,records
raw records,2313682
invalid timestamps,4
missing sensor ID,526
out-of-roster ID,9866
at least one missing channel,93879


**Technical:** 2,303,286 records have valid timestamps and roster IDs. Missing values or long gaps cannot be attributed to network loss without packet logs.  
**Plain language:** We can count what was recorded; we cannot tell why something is absent.

## Arrival rate and coverage
Endpoint days are partial. Even an interior day can include logging downtime; the charts describe the *recorded* workload and do not establish normal demand.


In [3]:
daily = events.groupby(events.timestamp.dt.floor('D')).size().rename('readings')
hourly = events.groupby(events.timestamp.dt.hour).size().reindex(range(24),fill_value=0)
per_mote = events.groupby('moteid').size().reindex(range(1,55),fill_value=0)
fig = px.line(x=daily.index,y=daily.values,labels={'x':'Source calendar day','y':'Logged readings'},title='1. Recorded volume by day'); fig.show()
fig = px.bar(x=hourly.index,y=hourly.values,labels={'x':'Source hour (across all dates)','y':'Logged readings'},title='2. Hour-of-day totals, without uptime correction'); fig.show()
fig = px.bar(x=per_mote.index,y=per_mote.values,labels={'x':'Sensor ID','y':'Logged readings'},title='3. Unequal sensor coverage'); fig.show()
interior = daily.iloc[1:-1]
display(Markdown(f'**Technical:** Interior-day volume ranges from {interior.min():,} to {interior.max():,}; sensor counts range from {per_mote.min():,} to {per_mote.max():,}. Low counts may reflect collection outages, and hour totals are not normalized by uptime.  \n**Plain language:** Recorded activity varies sharply, but we should check that sensors were working before calling a quiet day a low-traffic day.'))


**Technical:** Interior-day volume ranges from 279 to 101,207; sensor counts range from 35 to 65,693. Low counts may reflect collection outages, and hour totals are not normalized by uptime.  
**Plain language:** Recorded activity varies sharply, but we should check that sensors were working before calling a quiet day a low-traffic day.

## Interarrival gaps
Calculate intervals separately for each sensor. Report full-distribution quantiles alongside a histogram trimmed for legibility. This avoids letting gaps between *different* sensors masquerade as a single sensor's cadence.


In [4]:
gaps = events.groupby('moteid').timestamp.diff().dt.total_seconds().dropna()
gaps = gaps[gaps >= 0]
counts,edges = np.histogram(gaps[gaps <= gaps.quantile(.99)], bins=90)
fig = px.bar(x=(edges[:-1]+edges[1:])/2,y=counts,labels={'x':'Seconds between logged readings from one sensor','y':'Frequency'},title='4. Within-sensor gap frequency (upper 1% omitted from view)'); fig.show()
probabilities = np.linspace(0,1,101)
fig = px.line(x=gaps.quantile(probabilities).to_numpy(),y=probabilities,labels={'x':'Gap in seconds','y':'Cumulative proportion'},title='5. Gap quantiles, including the upper tail'); fig.show()
display(Markdown(f'**Technical:** Median within-sensor gap: {gaps.median():.1f} s; p95: {gaps.quantile(.95):.1f} s. These are conditional on observed records.  \n**Plain language:** A typical sensor reading was logged about half a minute after its previous one; some recorded gaps were much longer.'))


**Technical:** Median within-sensor gap: 33.2 s; p95: 149.0 s. These are conditional on observed records.  
**Plain language:** A typical sensor reading was logged about half a minute after its previous one; some recorded gaps were much longer.

## Interpretation and next dataset
These plots answer arrival and coverage questions for one historical wireless deployment. They do not measure east-west traffic. The primary follow-up is the [Alibaba 2021 microservice trace](https://github.com/alibaba/clusterdata/tree/master/cluster-trace-microservices-v2021): reconstruct caller-to-callee paths, stratify by call type, and disclose its 0.5% call-graph sample and large download size. The [assessment](assessment.md) lists evidence gaps. Continue in [Analysis II](02_models_and_experiments.ipynb).


In [5]:
# Daily counts need a coverage denominator before selecting a quiet replay.
coverage = events.groupby(events.timestamp.dt.floor('D')).agg(
    readings=('moteid','size'), active_motes=('moteid','nunique'))
display(coverage.sort_values('readings').head(6))
print('Observed timestamps:', events.timestamp.nunique(), 'of', len(events), 'accepted rows')
print('Within-mote gap p99 (s):', round(gaps.quantile(.99), 1))

,readings,active_motes
timestamp,,
2004-04-03,3,1
2004-04-02,279,3
2004-03-30,28020,40
2004-03-31,28217,34
2004-04-01,30636,25
2004-03-26,31256,44


Observed timestamps: 2303286 of 2303286 accepted rows
Within-mote gap p99 (s): 320.2


## Observations and deduction from the full run

The original archive has SHA-256 `d99288c8f406ca6604d359ceaa0d8adfffa79e7095061a1e27dc4399f48c7225`. Its 2,313,682 parsed rows yield 2,303,286 records with valid timestamps and roster IDs. The audit finds 4 invalid timestamps, 526 missing IDs, 9,866 out-of-roster IDs, and 93,879 rows with a missing sensor channel. These categories can overlap. Source timestamps have no asserted timezone. The valid interval is 28 February to 3 April 2004.

The median within-sensor interval is 33.2 s, p95 is 149.0 s, and p99 is 320.2 s. Counts span 35 to 65,693 per sensor. The 279-reading interior day (2 April) has only 3 active sensors; the final day has 3 readings from 1 sensor. Thus low daily volume is entangled with sensor availability. It would be unsound to label 2 April a naturally quiet demand profile without an uptime record. Select replay windows only after documenting active sensors and exclusions; retain the same event IDs and timing in every candidate run.

**Logical deduction:** This log establishes variation and coverage problems in a sensor ingestion workload. It cannot identify service edges, bytes, retries, or energy. Its strong coverage imbalance means an optimiser trained only on total event rate could mistake outages for optimization opportunities. The next observational input should identify real service paths and event compatibility; the next causal evidence must come from controlled service experiments. See [Analysis II](02_models_and_experiments.ipynb) for a batching-scope sensitivity check.

## Real service-path evidence: Alibaba temporal slices

The [published Alibaba trace](https://github.com/alibaba/clusterdata/tree/master/cluster-trace-microservices-v2021) samples call graphs at 0.5% and divides the twelve-hour period into 145 archives. We analyzed three complete five-minute shards: 0, 72, and 143. Together they contain 19,335,860 rows, 514,169 observed call graphs, and 14,960,728 logical calls after pairing records with the same trace and RPC ID. These totals cover the three selected slices only.

The busiest 1% of known-endpoint `(upstream, downstream, RPC type)` edges account for 68.28%, 68.32%, and 68.87% of calls within the three respective slices. The top-100 edge overlap is 88 (early/middle), 78 (early/late), and 85 (middle/late). This supports a route-targeted experiment and reduces concern that the early slice alone drove the pattern. It does **not** establish a twelve-hour or fleet-wide estimate. Missing usable endpoints affect 12.29%, 10.68%, and 18.33% of logical calls respectively, so quality changes over time. Paired-record type conflicts and missing parent RPC IDs also limit path reconstruction; missing parents include possible roots. The full [slice report](results/alibaba/report.md) and machine-readable metrics show these audits.

**Deduction:** Real edge activity is concentrated, but the trace does not reveal tenant, ordering, payload bytes, or whether events may legally share a batch. The service contract in Analysis II defines compatibility separately. Response-time values are source-recorded call times, not end-to-end task latency or a causal comparison of protocols.

In [6]:
import json
from pathlib import Path
slice_files=['results/alibaba/metrics.json','results/alibaba/shard72/metrics.json',
             'results/alibaba/shard143/metrics.json']
slices=[json.loads(Path(p).read_text(encoding='utf-8')) for p in slice_files]
slice_table=pd.DataFrame([{'shard':m['input'][0]['file'], 'minute_start':m['timestamp_min_ms']/60000,
                           'graphs':m['sampled_call_graphs'],'logical_calls':m['logical_calls'],
                           'known_edges':m['distinct_edges'],
                           'top_1pct_edge_share':m['top_1pct_edge_call_share'],
                           'missing_endpoint_share':m['calls_with_missing_endpoint']/m['logical_calls'],
                           'rpc_p95_ms':m['client_rt_by_type']['rpc']['p95_ms']} for m in slices])
display(slice_table)
edge_sample=pd.read_csv('results/alibaba/top_edges.csv')
display(edge_sample.head(10))
fig=px.bar(slice_table,x='shard',y='top_1pct_edge_share',
           title='Busiest 1% known-edge call share in three five-minute Alibaba slices')
fig.show()

,shard,minute_start,graphs,logical_calls,known_edges,top_1pct_edge_share,missing_endpoint_share,rpc_p95_ms
0,MSCallGraph_0.tar.gz,0.000033,131256,4641775,16100,0.682778,0.122907,169.0
1,MSCallGraph_72.tar.gz,360.000000,148031,4570686,16885,0.683164,0.106751,182.0
2,MSCallGraph_143.tar.gz,715.000000,234882,5748267,15941,0.688684,0.183301,141.0


,um,dm,rpctype,sampled_calls
0,af42b5e3e0eb334d38619733586d78d1414f6549f24d31...,b1dbd3a649a3cc790fa12573c9c1aa00988e07a8818a22...,mc,190230
1,75e56c8fbb9336eb4dd40f5f609d5344203d374d73fd0b...,84f9f68ef003a21288fffe8f9a09a5a29b05f4cc4229b8...,rpc,180551
2,95a6f7f8345e2eca31ee74ddc19d547e7fc0f5c8e65772...,95a6f7f8345e2eca31ee74ddc19d547e7fc0f5c8e65772...,rpc,134300
3,75e56c8fbb9336eb4dd40f5f609d5344203d374d73fd0b...,9a9e8613b6d7d1b573ba38dba19eec9c3cc460ab00cb49...,rpc,123766
4,75e56c8fbb9336eb4dd40f5f609d5344203d374d73fd0b...,091794afdcf1abbaf62404337ff72a38df96c5d8af8bd0...,rpc,97599
5,5cca70246befb1f4c9546d2912b9419dee54439218efa5...,b1dbd3a649a3cc790fa12573c9c1aa00988e07a8818a22...,mc,86578
6,75e56c8fbb9336eb4dd40f5f609d5344203d374d73fd0b...,75e56c8fbb9336eb4dd40f5f609d5344203d374d73fd0b...,userdefined,82335
7,95a6f7f8345e2eca31ee74ddc19d547e7fc0f5c8e65772...,6746853e39c5008988baf677bf27c846ee99173cafdec2...,mc,80907
8,4ab265f54516248ee8873be7d6441912456ce17e84f399...,fd6d86bd0fd550e717c1fdb82a33190a9fef216d87d535...,mc,65643
9,35114acfb54c54fb9618f23cd28bbc57c765f597df1409...,111624b7e31ece888671b8b90fe623f4c4ee2b33670640...,mc,52525


## Availability-aware replay windows and a second source

The [Azure Functions 2019 release](https://github.com/Azure/AzurePublicDataset/blob/master/AzureFunctionsDataset2019.md) provides 14 days of per-minute invocation counts for a sampled set of functions. It is not an inter-service call trace. We analyzed all 14 published invocation-day files (archive SHA-256 `aff8b3ca7240a41a109e4ee598e0a96e45fcb92e7b8395ac19cb3748cd260d89`), then selected 60-minute quiet, steady, and bursty windows by recorded rules in the [replay manifest](results/replay/manifest.json). Intel candidates require at least 40 distinct motes in every minute; this availability gate excludes apparent quiet periods with sparse sensor coverage.

The Intel windows contain 3,926, 4,557, and 4,582 observed events for quiet, steady, and bursty labels. Their peak-minute/mean-minute ratios are 1.15, 1.15, and 1.24, so even the selected Intel 'bursty' hour is only mildly variable. Azure's selected hours have peak/mean ratios of 1.16, 1.14, and 1.68. Azure counts are reduced by a documented 0.0001 scale for the testbed; the resulting minute-level replay counts are 2,995, 3,736, and 4,270. Intel uses its original event offsets and raw-row IDs. Azure individual event timestamps are unavailable, so events are deterministically spaced within each minute and labelled as constructed timing.

**Deduction:** The two traces provide different arrival regimes, but neither alone establishes representative internal service traffic. The source-specific route and timestamp limits remain explicit in the testbed contract.

In [7]:
replay_manifest=json.loads(Path('results/replay/manifest.json').read_text(encoding='utf-8'))
profiles=pd.read_csv('results/replay/selected_profiles.csv')
display(profiles.groupby('profile',as_index=False).agg(
    observed_count=('observed_count','sum'),replay_count=('replay_count','sum'),scale=('scale','first')))
fig=px.line(profiles,x='relative_minute',y='replay_count',color='profile',
            title='Selected replay counts per source minute (Azure scaled)')
fig.show()

,profile,observed_count,replay_count,scale
0,azure_bursty,42726264,4270,0.0001
1,azure_quiet,29937581,2995,0.0001
2,azure_steady,37363901,3736,0.0001
3,intel_bursty,4582,4582,1.0000
4,intel_quiet,3926,3926,1.0000
5,intel_steady,4557,4557,1.0000
